# Bài 5a (trang PDF 45, Germany) – Câu 6

**Sinh viên:** Trần Chí Nguyên  
**MSSV:** 6451071052

Notebook chạy độc lập trên Google Colab. Dữ liệu công khai được tải tự động khi chạy cell đầu tiên.

In [ ]:
import io, zipfile, requests, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("Trần Chí Nguyên - 6451071052")
FILE_ID = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
def load_retail():
    path = Path("/content/Online Retail.xlsx")
    if not path.exists():
        response = requests.get("https://drive.google.com/uc", params={"export": "download", "id": FILE_ID}, timeout=180)
        response.raise_for_status()
        path.write_bytes(response.content)
    return pd.read_excel(path, engine="openpyxl")
def clean_retail(data):
    data = data.copy()
    data["Description"] = data["Description"].astype("string").str.strip()
    data = data.dropna(subset=["InvoiceNo", "Description"])
    data["InvoiceNo"] = data["InvoiceNo"].astype(str).str.strip()
    data = data[~data["InvoiceNo"].str.contains("C", case=False, na=False)]
    return data
df_raw = load_retail()
print("Kích thước dữ liệu gốc:", df_raw.shape)
display(df_raw.head())

## Thực hiện yêu cầu

Cell dưới đây chứa lời giải cho câu này.

In [ ]:
%pip -q install mlxtend openpyxl
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
clean = clean_retail(df_raw); germany = clean.loc[clean["Country"].eq("Germany")]
basket = germany.pivot_table(index="InvoiceNo", columns="Description", values="Quantity", aggfunc="sum", fill_value=0)
basket = basket.drop(columns=["POSTAGE"], errors="ignore").gt(0).astype(pd.SparseDtype(bool, False))
freq_a = apriori(basket, min_support=0.05, use_colnames=True, low_memory=True)
freq_f = fpgrowth(basket, min_support=0.05, use_colnames=True)
rules_a = association_rules(freq_a, metric="confidence", min_threshold=.50)
rules_f = association_rules(freq_f, metric="confidence", min_threshold=.50)
print("Apriori: itemsets", len(freq_a), "rules", len(rules_a))
print("FP-Growth: itemsets", len(freq_f), "rules", len(rules_f))
print("Itemset equality:", {frozenset(x) for x in freq_a["itemsets"]} == {frozenset(x) for x in freq_f["itemsets"]})
print("Số hóa đơn Germany:", basket.shape[0], "| ngưỡng support count tương đương:", int(np.ceil(.05 * basket.shape[0])))